Recomandation Systems

ML systems used to get what a user would like.

Types of Recomendation System

1.) Content Based  - recomends based on the similarity of content.

2.) Collaborative Filltering - recomends based on the users interest.
eg- if A and B have similar likes, so if what A likes it will also be shown to B.

3.) Hybrid - Combination of both content and collaborative.

Project Flow

Data -> Preprocessing of Data -> Model Building ->  Webisite Building -> Deployment

Dataset: https://www.kaggle.com/datasets/bilalyussef/google-books-dataset?select=google_books_dataset.csv

In [40]:
import numpy as np
import pandas as pd
from html import unescape
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.porter import PorterStemmer
import pickle

1.) Data Preprocessing

In [33]:
books = pd.read_csv('google_books_1299.csv')

In [18]:
books

,Unnamed: 0,title,author,rating,voters,price,currency,description,publisher,page_count,generes,ISBN,language,published_date
0,0,Attack on Titan: Volume 13,Hajime Isayama,4.6,428,43.28,SAR,NO SAFE PLACE LEFT At great cost to the Garris...,Kodansha Comics,192,none,9781612626864,English,"Jul 31, 2014"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288,"Fiction , Mystery &amp, Detective , Cozy , Gen...",9780758272799,English,"Jul 1, 2007"
2,2,The Art of Super Mario Odyssey,Nintendo,3.9,9,133.85,SAR,Take a globetrotting journey all over the worl...,Dark Horse Comics,368,"Games &amp, Activities , Video &amp, Electronic",9781506713816,English,"Nov 5, 2019"
3,3,Getting Away Is Deadly: An Ellie Avery Mystery,Sara Rosett,4.0,10,26.15,SAR,"With swollen feet and swelling belly, pregnant...",Kensington Publishing Corp.,320,none,9781617734076,English,"Mar 1, 2009"
4,4,"The Painted Man (The Demon Cycle, Book 1)",Peter V. Brett,4.5,577,28.54,SAR,The stunning debut fantasy novel from author P...,HarperCollins UK,544,"Fiction , Fantasy , Dark Fantasy",9780007287758,English,"Jan 8, 2009"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1294,1294,Twas The Nightshift Before Christmas: Festive ...,Adam Kay,4.7,47,41.82,SAR,A short gift book of festive hospital diaries ...,Pan Macmillan,112,"Medical , Health Care Delivery",9781529018592,English,"Oct 17, 2019"
1295,1295,Why We Sleep: The New Science of Sleep and Dreams,Matthew Walker,4.8,52,46.85,SAR,'Astonishing ... an amazing book ... absolutel...,Penguin UK,368,"Psychology , Cognitive Psychology &amp, Cognition",9780141983776,English,"Sep 28, 2017"
1296,1296,How to Understand Business Finance: Edition 2,Bob Cinnamon,3.5,4,46.85,SAR,The modern marketplace is increasingly unpredi...,Kogan Page Publishers,176,none,9780749460211,English,"Apr 3, 2010"
1297,1297,Spider-Man: Kraven's Last Hunt,J. M. DeMatteis,4.6,74,43.28,SAR,"Collects Web of Spider-Man #31-32, Amazing Spi...",Marvel Entertainment,168,none,9781302377366,English,"Dec 10, 2014"


In [ ]:
books.info()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


1.1) Selecting Relevant Columns

Filtering the dataset down to the fields that will feed the content-based recommendation model.

In [34]:
books.rename(columns={'generes': 'genres'}, inplace=True)

In [ ]:
books.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


1.2) Selecting Feature Columns

Keep the text-rich columns that can describe a book for similarity search.

Columns kept: title, author, description, publisher, ISBN, genres

These fields provide enough signal to describe book content and metadata for our content-based approach.

In [35]:
books = books[['title', 'author', 'description', 'publisher', 'ISBN', 'genres']].copy()

In [21]:
books.head()

,title,author,description,publisher,ISBN,generes
0,Attack on Titan: Volume 13,Hajime Isayama,NO SAFE PLACE LEFT At great cost to the Garris...,Kodansha Comics,9781612626864,none
1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,9780758272799,"Fiction , Mystery &amp, Detective , Cozy , Gen..."
2,The Art of Super Mario Odyssey,Nintendo,Take a globetrotting journey all over the worl...,Dark Horse Comics,9781506713816,"Games &amp, Activities , Video &amp, Electronic"
3,Getting Away Is Deadly: An Ellie Avery Mystery,Sara Rosett,"With swollen feet and swelling belly, pregnant...",Kensington Publishing Corp.,9781617734076,none
4,"The Painted Man (The Demon Cycle, Book 1)",Peter V. Brett,The stunning debut fantasy novel from author P...,HarperCollins UK,9780007287758,"Fiction , Fantasy , Dark Fantasy"


1.3) Missing Data

Check for missing data, in this case we drop for any missing data

In [22]:
books.isnull().sum()

title          0
author         0
description    3
publisher      0
ISBN           0
generes        0
dtype: int64

In [36]:
books.dropna(subset=['title', 'author', 'description', 'genres'], inplace=True)
books.reset_index(drop=True, inplace=True)

1.4) Duplicate Data

checking for duplicate data, in this case also we remove the data

In [ ]:
books.duplicated(subset='title').sum()

np.int64(351)

In [37]:
books.drop_duplicates(subset='title', inplace=True)
books.reset_index(drop=True, inplace=True)

1.4) Editing Data

we have to edit the data to be used in a specific format

In [ ]:
books.iloc[0].genres

'Fiction , Fantasy , Action &amp, Adventure'

Feature Engineering

Tokenize descriptive text and normalize metadata so everything can be merged into a unified tag representation.

In [38]:
def clean_genres(genre_str):
    if pd.isna(genre_str):
        return []
    text = unescape(str(genre_str))
    tokens = [token.strip() for token in text.split(',')]
    cleaned = []
    for token in tokens:
        if not token or token.lower() == 'none':
            continue
        cleaned.append(token.replace(' ', ''))
    return cleaned

def tokenize_description(text):
    if pd.isna(text):
        return []
    return str(text).split()

def normalize_to_token(value):
    if pd.isna(value):
        return []
    value = str(value).strip()
    if not value:
        return []
    return [value.replace(' ', '')]

In [41]:
books['genres'] = books['genres'].apply(clean_genres)

In [42]:
books['description'] = books['description'].apply(tokenize_description)

In [43]:
books['author'] = books['author'].apply(normalize_to_token)
books['publisher'] = books['publisher'].apply(normalize_to_token)
books['ISBN'] = books['ISBN'].apply(normalize_to_token)

In [44]:
def assemble_tags(row):
    tokens = []
    for column in ['description', 'genres', 'author', 'publisher', 'ISBN']:
        tokens.extend(row[column])
    return tokens

books['tags'] = books.apply(assemble_tags, axis=1)
books[['title','tags']].head()

,title,tags
0,Attack on Titan: Volume 13,"[NO, SAFE, PLACE, LEFT, At, great, cost, to, t..."
1,Antiques Roadkill: A Trash 'n' Treasures Mystery,"[Determined, to, make, a, new, start, in, her,..."
2,The Art of Super Mario Odyssey,"[Take, a, globetrotting, journey, all, over, t..."
3,Getting Away Is Deadly: An Ellie Avery Mystery,"[With, swollen, feet, and, swelling, belly,, p..."
4,"The Painted Man (The Demon Cycle, Book 1)","[The, stunning, debut, fantasy, novel, from, a..."


Inspect Tokens

Ensure the generated tokens look reasonable across different books.

In [ ]:
books[['title','genres']].head()

In [ ]:
books[['title','description']].head()

In [ ]:
books.sample(5)[['title','tags']]

,genres,id,keywords,original_title,overview,cast,crew
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...",Avatar,"In the 22nd century, a paraplegic Marine is di...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug abuse, exotic island, east india ...",Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Johnny Depp, Orlando Bloom, Keira Knightley]","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,"[Action, Adventure, Crime]",206647,"[spy, based on novel, secret agent, sequel, mi...",Spectre,A cryptic message from Bond’s past sends him o...,"[Daniel Craig, Christoph Waltz, Léa Seydoux]","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,"[Action, Crime, Drama, Thriller]",49026,"[dc comics, crime fighter, terrorist, secret i...",The Dark Knight Rises,Following the death of District Attorney Harve...,"[Christian Bale, Michael Caine, Gary Oldman]","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,"[Action, Adventure, Science Fiction]",49529,"[based on novel, mars, medallion, space travel...",John Carter,"John Carter is a war-weary, former military ca...","[Taylor Kitsch, Lynn Collins, Samantha Morton]","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


Editing the column of Crew

In [ ]:
books['tags'].str.split().map(len).describe()

In [ ]:
books[['title','tags']].sample(3)

In [ ]:
books[['title','author','publisher']].head()

,genres,id,keywords,original_title,overview,cast,crew,creww
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...",Avatar,"In the 22nd century, a paraplegic Marine is di...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",[James Cameron]
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug abuse, exotic island, east india ...",Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Johnny Depp, Orlando Bloom, Keira Knightley]","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",[Gore Verbinski]
2,"[Action, Adventure, Crime]",206647,"[spy, based on novel, secret agent, sequel, mi...",Spectre,A cryptic message from Bond’s past sends him o...,"[Daniel Craig, Christoph Waltz, Léa Seydoux]","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",[Sam Mendes]
3,"[Action, Crime, Drama, Thriller]",49026,"[dc comics, crime fighter, terrorist, secret i...",The Dark Knight Rises,Following the death of District Attorney Harve...,"[Christian Bale, Michael Caine, Gary Oldman]","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",[Christopher Nolan]
4,"[Action, Adventure, Science Fiction]",49529,"[based on novel, mars, medallion, space travel...",John Carter,"John Carter is a war-weary, former military ca...","[Taylor Kitsch, Lynn Collins, Samantha Morton]","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",[Andrew Stanton]


Finalize Tag Corpus

Convert tokens back to a normalized string that will feed the vectorizer.

In [45]:
books['tags'] = books['tags'].apply(lambda x: [token.lower() for token in x])

In [ ]:
books[['title','tags']].head()

,genres,id,keywords,original_title,overview,cast,crew,creww
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...",Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",[James Cameron]
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug abuse, exotic island, east india ...",Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Johnny Depp, Orlando Bloom, Keira Knightley]","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",[Gore Verbinski]
2,"[Action, Adventure, Crime]",206647,"[spy, based on novel, secret agent, sequel, mi...",Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Daniel Craig, Christoph Waltz, Léa Seydoux]","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",[Sam Mendes]
3,"[Action, Crime, Drama, Thriller]",49026,"[dc comics, crime fighter, terrorist, secret i...",The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Christian Bale, Michael Caine, Gary Oldman]","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",[Christopher Nolan]
4,"[Action, Adventure, Science Fiction]",49529,"[based on novel, mars, medallion, space travel...",John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Taylor Kitsch, Lynn Collins, Samantha Morton]","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",[Andrew Stanton]


Normalize Tags

Join tokens and lowercase them so each book has a single processed document string.

In [46]:
books['tags'] = books['tags'].apply(lambda x: " ".join(x))

In [47]:
books['tags'] = books['tags'].apply(lambda x: x.lower())

In [ ]:
books[['title','tags']].head()

2.) Feature Matrix

Vectorize the processed tag text so we can measure similarity between books.

In [48]:
new_df = books[['title','tags']].copy()

In [ ]:
new_df.head()

,genres,id,keywords,original_title,overview,cast,crew,creww,tags
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...",Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[SamWorthington, ZoeSaldana, SigourneyWeaver]","[[, {, "", c, r, e, d, i, t, _, i, d, "", :, , ""...",[James Cameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,"[Adventure, Fantasy, Action]",285,"[ocean, drugabuse, exoticisland, eastindiatrad...",Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]","[[, {, "", c, r, e, d, i, t, _, i, d, "", :, , ""...",[Gore Verbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,"[Action, Adventure, Crime]",206647,"[spy, basedonnovel, secretagent, sequel, mi6, ...",Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[DanielCraig, ChristophWaltz, LéaSeydoux]","[[, {, "", c, r, e, d, i, t, _, i, d, "", :, , ""...",[Sam Mendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,"[Action, Crime, Drama, Thriller]",49026,"[dccomics, crimefighter, terrorist, secretiden...",The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[ChristianBale, MichaelCaine, GaryOldman]","[[, {, "", c, r, e, d, i, t, _, i, d, "", :, , ""...",[Christopher Nolan],"[Following, the, death, of, District, Attorney..."
4,"[Action, Adventure, ScienceFiction]",49529,"[basedonnovel, mars, medallion, spacetravel, p...",John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[TaylorKitsch, LynnCollins, SamanthaMorton]","[[, {, "", c, r, e, d, i, t, _, i, d, "", :, , ""...",[Andrew Stanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [ ]:
new_df.info()

1.6) Stemming

Reduce inflected words to their base form to keep vocabulary compact.

In [49]:
ps = PorterStemmer()

def stem(text):
    return " ".join(ps.stem(token) for token in text.split())

new_df['tags'] = new_df['tags'].apply(stem)

In [50]:
new_df.head()

,title,tags
0,Attack on Titan: Volume 13,no safe place left at great cost to the garris...
1,Antiques Roadkill: A Trash 'n' Treasures Mystery,determin to make a new start in her quaint hom...
2,The Art of Super Mario Odyssey,take a globetrot journey all over the world--a...
3,Getting Away Is Deadly: An Ellie Avery Mystery,"with swollen feet and swell belly, pregnant el..."
4,"The Painted Man (The Demon Cycle, Book 1)",the stun debut fantasi novel from author peter...


2.) Vectorization

Process of converting text to the numerical value.

we will be using tecchnique Bag of Words.

In this we will cconvert the tags to the vector.

Vectors are marked in a graph and when a user selcct a movie, all the surrounding movie will be selcted for the recomendation.

Bag of Words

In this we combine all the tags into a single string and take the most common words.

Then we compile a table of all the words that occur and note their frequency for each movie.

This will convert the tags to the vectors

Note: Stop words will not be considered. we have to remove the stop words before hand.

In [51]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

In [52]:
print(f"Vector shape: {vectors.shape}")
print(f"Number of books: {vectors.shape[0]}")
print(f"Number of features: {vectors.shape[1]}")

Vector shape: (243, 5000)
Number of books: 243
Number of features: 5000


In [ ]:
cv.get_feature_names_out()[:50]

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      shape=(5000,), dtype=object)

3.) Similarity Computation

Calculate cosine similarity between book vectors to find similar books.

In [53]:
similarity = cosine_similarity(vectors)
similarity

array([[1.        , 0.04372172, 0.01274532, ..., 0.03857584, 0.01416198,
        0.02837522],
       [0.04372172, 1.        , 0.03009135, ..., 0.03035884, 0.03064971,
        0.05861898],
       [0.01274532, 0.03009135, 1.        , ..., 0.05309942, 0.18194289,
        0.01952916],
       ...,
       [0.03857584, 0.03035884, 0.05309942, ..., 1.        , 0.04261219,
        0.01970276],
       [0.01416198, 0.03064971, 0.18194289, ..., 0.04261219, 1.        ,
        0.04339971],
       [0.02837522, 0.05861898, 0.01952916, ..., 0.01970276, 0.04339971,
        1.        ]], shape=(243, 243))

4.) Recommendation Function

Create the main function to generate book recommendations based on similarity scores.

In [54]:
def recommend(book_title):
    if book_title not in new_df['title'].values:
        print(f"Book '{book_title}' not found in database.")
        return
    
    book_index = new_df[new_df['title'] == book_title].index[0]
    distances = similarity[book_index]
    books_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    print(f"\nRecommendations for '{book_title}':\n")
    for i, (idx, score) in enumerate(books_list, 1):
        print(f"{i}. {new_df.iloc[idx]['title']} (similarity: {score:.3f})")

In [55]:
recommend('Attack on Titan: Volume 13')


Recommendations for 'Attack on Titan: Volume 13':

1. The Tower of the Swallow: Witcher 6 (similarity: 0.176)
2. Auberon (similarity: 0.157)
3. Getting Things Done: The Art of Stress-Free Productivity (similarity: 0.125)
4. A Dance with Dragons: A Song of Ice and Fire: Book Five (similarity: 0.123)
5. The Weight of Honor (Kings and Sorcerers--Book 3) (similarity: 0.118)


In [60]:
recommend('The Law of Success in Sixteen Lessons')


Recommendations for 'The Law of Success in Sixteen Lessons':

1. The Last Wife: An absolutely gripping and emotional page turner with a brilliant twist (similarity: 0.454)
2. Summary: Think and Grow Rich (similarity: 0.359)
3. Influence: The Psychology of Persuasion (similarity: 0.342)
4. Getting Things Done: The Art of Stress-Free Productivity (similarity: 0.303)
5. Homecoming (A Chloe Fine Psychological Suspense Mystery—Book 5) (similarity: 0.300)


In [57]:
# Display available books
print(f"Total books available: {len(new_df)}\n")
print("Sample book titles:")
for idx, title in enumerate(new_df['title'].head(10), 1):
    print(f"{idx}. {title}")

Total books available: 243

Sample book titles:
1. Attack on Titan: Volume 13
2. Antiques Roadkill: A Trash 'n' Treasures Mystery
3. The Art of Super Mario Odyssey
4. Getting Away Is Deadly: An Ellie Avery Mystery
5. The Painted Man (The Demon Cycle, Book 1)
6. A Feast for Crows (A Song of Ice and Fire, Book 4)
7. God of War: The Official Novelization
8. Edgedancer: From the Stormlight Archive
9. Blood, Sweat, and Pixels: The Triumphant, Turbulent Stories Behind How Video Games Are Made
10. Twas The Nightshift Before Christmas: Festive hospital diaries from the author of million-copy hit This is Going to Hurt


5.) Export Data

Save the processed data and similarity matrix for deployment.

In [58]:
pickle.dump(new_df, open('books.pkl','wb'))
print("Books data saved to books.pkl")

Books data saved to books.pkl


In [59]:
pickle.dump(similarity, open('similarity.pkl','wb'))
print("Similarity matrix saved to similarity.pkl")

Similarity matrix saved to similarity.pkl
